In [ ]:
# STEP 1: Now, let's look at just the subset of active users.

import pandas as pd
import numpy as np

# Load Active sheet 
df2 = pd.read_excel(
    "../mock-data/Active_Inactive_Non-Diala_Mock.xlsx",
    sheet_name="Active_Users"
)
print(f"Raw shape: {df2.shape}")
print(f"Sheet columns: {list(df2.columns)}")

# Drop completely empty rows 
df2 = df2.dropna(how="all")
print(f"Shape after dropping empty rows: {df2.shape}")

# Coercing numeric columns immediately after load 
# I did this before any string operations!
numeric_cols = [
    "AGE", "AOS", "Diab_Duration", "Followup_Duration_BL_FU",
    "HbA1c_BL", "BMI_BL", "Waist_BL", "HDL_BL", "TGL_BL",
    "BP_Dia_BL", "BP_Sys_BL", "Serum_Cholesterol_BL",
    "HbA1c_FU", "BMI_FU", "Waist_FU", "HDL_FU", "TGL_FU",
    "BP_Dia_Fu", "BP_Sys_FU", "Serum_Cholesterol_FU",
    "VisitCount", "Credit_Score"
]
for col in numeric_cols:
    if col in df2.columns:
        df2[col] = pd.to_numeric(df2[col], errors="coerce")

# Clean string columns only 
# I needed to strip whitespace and replace empty-looking values with NaN
# Only touches non-numeric columns so AOS, Diab_Duration etc. are safe
replace_vals = ["", " ", "NA", "N/A", "na", "n/a", "NIL", "nil",
                "None", "none", "NULL", "null", "-", "--", "nan"]

string_cols = df2.select_dtypes(include="object").columns
for col in string_cols:
    df2[col] = df2[col].astype(str).str.strip()
    df2[col] = df2[col].replace(replace_vals + ["nan", "NaN"], np.nan)

# Here, I just wanted to verify key columns before recoding 
print(f"\nAOS     — dtype: {df2['AOS'].dtype} | NaN: {df2['AOS'].isna().sum()} / {len(df2)}")
print(f"Diab_Duration — dtype: {df2['Diab_Duration'].dtype} | NaN: {df2['Diab_Duration'].isna().sum()} / {len(df2)}")

# 1. Kuppuswamy Occupation - similar to what I was doing before!
kupp_map = {
    "Politician": 10, "Manager": 10, "Central Government Service": 10,
    "State Government Service": 10, "Tahsildar": 10,
    "Doctor": 9, "Doctor-Dentist": 9, "Doctor-General Physician": 9,
    "Doctor-Ophthalmologist": 9, "Doctor-Homeopathist": 9,
    "Doctor-Pediatrician": 9, "Doctor-Gynecologist": 9,
    "Doctor-Surgeon": 9, "Doctor-Neurologist": 9,
    "Doctor-Siddha": 9, "Doctor-Ayurvedic": 9,
    "Advocate": 9, "Architecture": 9, "AUDITOR": 9,
    "Engineer": 9, "IT Professional": 9,
    "Professor / Teacher / Education": 9, "Doctorate": 9,
    "Reporter": 8, "Accounts/Finance": 8, "Bank": 8,
    "IT Employee": 8, "Supervisor": 8, "Armed Forces": 8, "Police": 8,
    "Clerk": 7, "Government": 7,
    "Business": 6, "Self Employed": 6, "Fashion / Saloon": 6,
    "Retired Employee": 6, "Father In Church": 6, "Priest": 6, "Social Service": 6,
    "Farmer / Agriculture": 5,
    "Private Sector": 4,
    "Driver": 3, "Courier": 3,
    "Daily wages": 2,
    "Housewife": 1, "Retired": 1, "Armed Forces-Retired": 1, "Student": 1,
}

df2["Kupp_Occupation"] = df2["Occupation"].map(kupp_map)

n_before = len(df2)
df2 = df2.dropna(subset=["Kupp_Occupation"])
print(f"\nRemoved (missing/unmapped occupation): {n_before - len(df2)} rows")
print(f"Remaining: {len(df2)} rows")

# 2. Gender 
# 1 = Male | 2 = Female
df2["Gender"] = df2["Gender"].map({"M": "Male", "F": "Female"})
df2["Gender_Code"] = df2["Gender"].map({"Male": 1, "Female": 2})

# 3. Age Group 
# 0=<30 | 1=30–39 | 2=40–49 | 3=50–59 | 4=≥60
# Ref: Sathish et al., Indian J Med Res, 2010
def age_group(age):
    if pd.isna(age):  return np.nan
    elif age < 30:    return 0
    elif age < 40:    return 1
    elif age < 50:    return 2
    elif age < 60:    return 3
    else:             return 4

df2["Age_Group"] = df2["AGE"].apply(age_group)
age_labels = {0: "<30", 1: "30–39", 2: "40–49", 3: "50–59", 4: "≥60"}

# 4. Engagement_category → numeric 
# 1 = Low | 2 = Moderate | 3 = Champion
df2["Engagement_Code"] = df2["Engagement_category"].map({"Low": 1, "Moderate": 2, "Champion": 3})

# Final NaN summary: I always likeed having a summary to make sure columns were in order. 
print("\nNaN per column (post cleaning):")
print(df2.isna().sum().to_string())

print(f"\nFinal shape: {df2.shape}")
print("\nCoding reference:")
print("  Gender_Code         : 1=Male | 2=Female")
print("  Age_Group           : 0=<30 | 1=30–39 | 2=40–49 | 3=50–59 | 4=≥60")
print("  Engagement_Code     : 1=Low | 2=Moderate | 3=Champion")
print("  Kupp_Occupation     : 1–10 (Kuppuswamy scale)")

print(df2[["MRNO", "Gender", "Gender_Code", "AGE", "Age_Group",
           "AOS", "Diab_Duration",
           "Engagement_category", "Engagement_Code",
           "Occupation", "Kupp_Occupation"]].head(10).to_string())

In [ ]:
# Step 2: Missingness Check; basically if more than 20% of the values are missing, I exclude the variable from analysis.

outcome_vars = {
    "HbA1c":             ("HbA1c_BL",            "HbA1c_FU"),
    "HDL":               ("HDL_BL",               "HDL_FU"),
    "TGL":               ("TGL_BL",               "TGL_FU"),
    "Serum_Cholesterol": ("Serum_Cholesterol_BL", "Serum_Cholesterol_FU"),
}

THRESHOLD = 0.20
n_total = len(df2)

print("=" * 60)
print(f"MISSINGNESS REPORT  (n={n_total}, threshold={int(THRESHOLD*100)}%)")
print("=" * 60)
print(f"{'Variable':<22} {'BL Missing':>12} {'FU Missing':>12} {'Include?':>10}")
print("-" * 60)

include_vars = []
exclude_vars = []

for name, (bl_col, fu_col) in outcome_vars.items():
    bl_miss = df2[bl_col].isna().sum() / n_total if bl_col in df2.columns else 1.0
    fu_miss = df2[fu_col].isna().sum() / n_total if fu_col in df2.columns else 1.0
    worst   = max(bl_miss, fu_miss)
    flag    = "✓ YES" if worst < THRESHOLD else "✗ EXCLUDE"

    print(f"{name:<22} {bl_miss*100:>10.1f}%  {fu_miss*100:>10.1f}%  {flag:>10}")

    if worst < THRESHOLD:
        include_vars.append(name)
    else:
        exclude_vars.append(name)

print("=" * 60)
print(f"\n✓ INCLUDED ({len(include_vars)}): {include_vars}")
print(f"✗ EXCLUDED ({len(exclude_vars)}): {exclude_vars}")

In [ ]:
# Step 3: Finalise dataframe & calculate deltas 

# Keep only relevant columns 
keep_cols = [
    "MRNO", "Gender", "Gender_Code",
    "AGE", "Age_Group",
    "AOS", "Diab_Duration",
    "Occupation", "Kupp_Occupation",
    "Engagement_category", "Engagement_Code", "Credit_Score",
    "VisitCount",
    # Baseline values
    "HbA1c_BL", "HDL_BL", "TGL_BL", "Serum_Cholesterol_BL",
    # Follow-up values
    "HbA1c_FU", "HDL_FU", "TGL_FU", "Serum_Cholesterol_FU",
]

df2 = df2[[c for c in keep_cols if c in df2.columns]].copy()
print(f"Columns kept: {df2.shape[1]}")
print(f"Columns in df2: {list(df2.columns)}")

# Calculate deltas (FU - BL); basically follow-up minus baseline.
# HbA1c, TGL, Serum_Cholesterol : negative delta = improvement
# HDL                            : positive delta = improvement
df2["Delta_HbA1c"]             = df2["HbA1c_FU"]             - df2["HbA1c_BL"]
df2["Delta_HDL"]               = df2["HDL_FU"]               - df2["HDL_BL"]
df2["Delta_TGL"]               = df2["TGL_FU"]               - df2["TGL_BL"]
df2["Delta_Serum_Cholesterol"] = df2["Serum_Cholesterol_FU"] - df2["Serum_Cholesterol_BL"]

# NaN note
# Rows are NOT dropped for missing individual outcomes
# Delta = NaN automatically when BL or FU is missing
# Those rows are skipped per-variable during analysis via dropna()
# Only rows with NO complete BL+FU pair across all four outcomes dropped

n_before = len(df2)

has_any_complete = (
    (df2["HbA1c_BL"].notna()             & df2["HbA1c_FU"].notna())             |
    (df2["HDL_BL"].notna()               & df2["HDL_FU"].notna())               |
    (df2["TGL_BL"].notna()               & df2["TGL_FU"].notna())               |
    (df2["Serum_Cholesterol_BL"].notna() & df2["Serum_Cholesterol_FU"].notna())
)

df2 = df2[has_any_complete].copy()
print(f"\nRows dropped (no complete BL+FU for any outcome): {n_before - len(df2)}")
print(f"Final analysis n: {len(df2)}")

# Delta summary of stats
delta_cols = ["Delta_HbA1c", "Delta_HDL", "Delta_TGL", "Delta_Serum_Cholesterol"]

print("\n" + "=" * 60)
print("DELTA SUMMARY  (FU - BL)")
print("  HbA1c / TGL / Serum_Cholesterol : negative = improvement")
print("  HDL                             : positive = improvement")
print("=" * 60)
print(df2[delta_cols].describe().round(3).to_string())

# Missingness on deltas; same 20% threshold as before
print("\nMissing delta values:")
for col in delta_cols:
    n_miss = df2[col].isna().sum()
    pct    = n_miss / len(df2) * 100
    print(f"  {col:<35} {n_miss} missing ({pct:.1f}%)")

# Final head; sorry if this is repetitive, but I like to see the final dataframe before analysis.
print("\n✔ Step 3 complete — df2 ready for analysis.")
print(df2[["MRNO", "Gender", "Gender_Code", "Age_Group", "Kupp_Occupation",
           "Engagement_category", "Engagement_Code", "Credit_Score", "VisitCount","Diab_Duration", "AOS",
           "HbA1c_BL", "HbA1c_FU", "Delta_HbA1c",
           "HDL_BL",   "HDL_FU",   "Delta_HDL",
           "TGL_BL",   "TGL_FU",   "Delta_TGL",
           "Serum_Cholesterol_BL", "Serum_Cholesterol_FU",
           "Delta_Serum_Cholesterol"]].head(10).to_string())

In [ ]:
# Step 4: Descriptive Profile of Active Users

print("=" * 60)
print("DESCRIPTIVE PROFILE — ACTIVE USERS")
print("=" * 60)

# Continuous variables 
continuous = {
    "Age (years)":               "AGE",
    "Age of Onset (years)":      "AOS",
    "Diabetes Duration (years)": "Diab_Duration",
    "Kuppuswamy SES Score":      "Kupp_Occupation",
    "Credit Score":              "Credit_Score",
    "Visit Count":               "VisitCount",
    "HbA1c BL (%)":             "HbA1c_BL",
    "HDL BL (mg/dL)":           "HDL_BL",
    "TGL BL (mg/dL)":           "TGL_BL",
    "Serum Cholesterol BL":      "Serum_Cholesterol_BL",
}

print(f"\n{'Variable':<30} {'n':>6} {'Mean':>8} {'SD':>8} {'Min':>8} {'Max':>8}")
print("-" * 72)
for label, col in continuous.items():
    if col in df2.columns:
        n    = df2[col].notna().sum()
        mean = df2[col].mean()
        sd   = df2[col].std()
        mn   = df2[col].min()
        mx   = df2[col].max()
        print(f"{label:<30} {n:>6} {mean:>8.2f} {sd:>8.2f} {mn:>8.2f} {mx:>8.2f}")

# Categorical variables
print("\n" + "=" * 60)
print("CATEGORICAL VARIABLES")
print("=" * 60)

# Gender
print("\nGender:")
gender_counts = df2["Gender"].value_counts(dropna=True)
for val, count in gender_counts.items():
    pct = count / len(df2) * 100
    print(f"  {val:<15} n={count:>4}  ({pct:.1f}%)")

# Age Group
print("\nAge Group:")
age_counts = df2["Age_Group"].value_counts(dropna=True).sort_index()
for val, count in age_counts.items():
    pct = count / len(df2) * 100
    print(f"  {age_labels[val]:<15} n={count:>4}  ({pct:.1f}%)")

# Engagement Category
print("\nEngagement Category:")
eng_order  = ["Low", "Moderate", "Champion"]
eng_counts = df2["Engagement_category"].value_counts(dropna=True)
for val in eng_order:
    if val in eng_counts:
        count = eng_counts[val]
        pct   = count / len(df2) * 100
        print(f"  {val:<15} n={count:>4}  ({pct:.1f}%)")

# SES — granular (1–10)
print("\nSES Score — Granular (Kuppuswamy 1–10):")
kupp_labels = {
    1:  "1  — Unskilled/Housewife",
    2:  "2  — Daily wages",
    3:  "3  — Driver/Courier",
    4:  "4  — Private Sector",
    5:  "5  — Farmer/Agriculture",
    6:  "6  — Business/Self-Employed",
    7:  "7  — Clerk/Government",
    8:  "8  — Bank/Supervisor/Armed Forces",
    9:  "9  — Doctor/Engineer/Advocate",
    10: "10 — Politician/Manager/Senior Govt",
}
kupp_counts = df2["Kupp_Occupation"].value_counts(dropna=True).sort_index()
for val, count in kupp_counts.items():
    pct   = count / len(df2) * 100
    label = kupp_labels.get(int(val), str(val))
    print(f"  {label:<40} n={count:>4}  ({pct:.1f}%)")

# SES — binned (Low / Middle / High)
print("\nSES Group — Binned:")
df2["SES_Group"] = pd.cut(
    df2["Kupp_Occupation"],
    bins=[0, 4, 7, 10],
    labels=["Low (1–4)", "Middle (5–7)", "High (8–10)"]
)
ses_counts = df2["SES_Group"].value_counts(dropna=True).sort_index()
for val, count in ses_counts.items():
    pct = count / len(df2) * 100
    print(f"  {str(val):<15} n={count:>4}  ({pct:.1f}%)")


In [ ]:
# Step 5: Clinical Outcomes BL → FU (baseline minus follow-up)
from scipy import stats
import numpy as np

# Cohen's D function (manual) 
def cohens_d_paired(a, b):
    diff = a - b
    return diff.mean() / diff.std()

print("=" * 70)
print("STEP 5: CLINICAL OUTCOMES — DID THEY IMPROVE? (BL → FU)")
print("=" * 70)

outcomes = {
    "HbA1c":             ("HbA1c_BL",            "HbA1c_FU",            "Delta_HbA1c",            "negative"),
    "HDL":               ("HDL_BL",               "HDL_FU",               "Delta_HDL",               "positive"),
    "TGL":               ("TGL_BL",               "TGL_FU",               "Delta_TGL",               "negative"),
    "Serum_Cholesterol": ("Serum_Cholesterol_BL", "Serum_Cholesterol_FU", "Delta_Serum_Cholesterol", "negative"),
}

results = []

for name, (bl_col, fu_col, delta_col, direction) in outcomes.items():

    # Use only rows with both BL and FU present 
    sub = df2[[bl_col, fu_col, delta_col]].dropna()
    n   = len(sub)

    if n < 3:
        print(f"\n{name}: insufficient data (n={n}), skipping.")
        continue

    bl_mean = sub[bl_col].mean()
    bl_sd   = sub[bl_col].std()
    fu_mean = sub[fu_col].mean()
    fu_sd   = sub[fu_col].std()
    delta   = sub[delta_col].mean()

    # Normality check on delta (Shapiro-Wilk) 
    delta_vals = sub[delta_col]
    if len(delta_vals) > 5000:
        delta_sample = delta_vals.sample(5000, random_state=42)
    else:
        delta_sample = delta_vals

    shapiro_stat, shapiro_p = stats.shapiro(delta_sample)
    normal = shapiro_p > 0.05

    # Paired test 
    if normal:
        test_name = "Paired t-test"
        t_stat, p_val = stats.ttest_rel(sub[bl_col], sub[fu_col])
    else:
        test_name = "Wilcoxon"
        t_stat, p_val = stats.wilcoxon(sub[bl_col], sub[fu_col])

    # Cohen's D (manual paired) 
    cd = cohens_d_paired(sub[bl_col], sub[fu_col])

    # Significance flag 
    sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"

    # Improvement flag 
    if direction == "negative":
        improved = "✓ improved" if delta < 0 else "✗ worsened"
    else:
        improved = "✓ improved" if delta > 0 else "✗ worsened"

    results.append({
        "Variable":   name,
        "n":          n,
        "BL Mean±SD": f"{bl_mean:.2f} ± {bl_sd:.2f}",
        "FU Mean±SD": f"{fu_mean:.2f} ± {fu_sd:.2f}",
        "Delta":      round(delta, 3),
        "Test":       test_name,
        "Stat":       round(t_stat, 3),
        "p_value":    round(p_val, 4),
        "Sig":        sig,
        "Cohens_D":   round(abs(cd), 3),
        "Direction":  improved,
        "Normality":  f"SW p={shapiro_p:.3f} ({'normal' if normal else 'non-normal'})",
    })

    # Print result!!
    print(f"\n{'─' * 70}")
    print(f"{name} {sig}  |  {improved}  |  n={n}")
    print(f"  BL:        {bl_mean:.2f} ± {bl_sd:.2f}")
    print(f"  FU:        {fu_mean:.2f} ± {fu_sd:.2f}")
    print(f"  Delta:     {delta:.3f}")
    print(f"  Test:      {test_name} | stat={t_stat:.3f} | p={p_val:.4f} {sig}")
    print(f"  Cohen's D: {abs(cd):.3f}")
    print(f"  Normality: SW p={shapiro_p:.3f} ({'normal' if normal else 'non-normal'})")

# Summary table
print(f"\n{'=' * 70}")
print("SUMMARY TABLE")
print(f"{'=' * 70}")
print(f"{'Variable':<22} {'n':>5} {'BL Mean±SD':<18} {'FU Mean±SD':<18} "
      f"{'Delta':>8} {'p-value':>8} {'Sig':>5} {'CohenD':>8} {'Result'}")
print("-" * 105)
for r in results:
    cd  = r["Cohens_D"]
    pv  = r["p_value"]
    print(f"{r['Variable']:<22} {r['n']:>5} {r['BL Mean±SD']:<18} {r['FU Mean±SD']:<18} "
          f"{r['Delta']:>8} {pv:>8} {r['Sig']:>5} {cd:>8} {r['Direction']}")


In [ ]:
# Step 6: Does SES predict clinical improvement? 
from scipy import stats
from itertools import combinations
import matplotlib.pyplot as plt
import numpy as np

print("=" * 70)
print("STEP 6: SES vs CLINICAL IMPROVEMENT")
print("=" * 70)

delta_outcomes = {
    "HbA1c":             ("Delta_HbA1c",            "negative"),
    "HDL":               ("Delta_HDL",               "positive"),
    "TGL":               ("Delta_TGL",               "negative"),
    "Serum_Cholesterol": ("Delta_Serum_Cholesterol", "negative"),
}

ses_order = ["Low (1–4)", "Middle (5–7)", "High (8–10)"]

# Make sure SES_Group exists 
if "SES_Group" not in df2.columns:
    df2["SES_Group"] = pd.cut(
        df2["Kupp_Occupation"],
        bins=[0, 4, 7, 10],
        labels=["Low (1–4)", "Middle (5–7)", "High (8–10)"]
    )

# Manual Dunn's post-hoc 
def dunn_posthoc(data, group_col, val_col, groups):
    from scipy.stats import rankdata
    all_vals   = data[val_col].values
    all_groups = data[group_col].values
    n_total    = len(all_vals)
    ranks      = rankdata(all_vals)
    pairs      = list(combinations(groups, 2))
    results    = {}
    for g1, g2 in pairs:
        idx1 = all_groups == g1
        idx2 = all_groups == g2
        n1, n2 = idx1.sum(), idx2.sum()
        r1, r2 = ranks[idx1].mean(), ranks[idx2].mean()
        se = np.sqrt((n_total * (n_total + 1) / 12.0) * (1.0 / n1 + 1.0 / n2))
        z  = (r1 - r2) / se
        p  = 2 * stats.norm.sf(abs(z))
        results[(g1, g2)] = p
    n_pairs   = len(pairs)
    corrected = {k: min(v * n_pairs, 1.0) for k, v in results.items()}
    return corrected

step6_results = []

for name, (delta_col, direction) in delta_outcomes.items():

    sub = df2[["SES_Group", "Kupp_Occupation", delta_col]].dropna().copy()
    sub["SES_Group"] = sub["SES_Group"].astype(str)
    n   = len(sub)

    print(f"\n{'─' * 70}")
    print(f"{name}  |  n={n}")

    # Group means 
    print(f"\n  Group means:")
    for grp in ses_order:
        grp_data = sub[sub["SES_Group"] == grp][delta_col]
        if len(grp_data) > 0:
            print(f"    {grp:<18} n={len(grp_data):>4}  "
                  f"mean={grp_data.mean():>7.3f}  sd={grp_data.std():>7.3f}")

    # Kruskal-Wallis 
    groups = [
        sub[sub["SES_Group"] == g][delta_col].dropna().values
        for g in ses_order
    ]
    groups = [g for g in groups if len(g) >= 3]

    if len(groups) < 2:
        print("  Insufficient groups for Kruskal-Wallis, skipping.")
        continue

    kw_stat, kw_p = stats.kruskal(*groups)
    kw_sig = "***" if kw_p < 0.001 else "**" if kw_p < 0.01 else "*" if kw_p < 0.05 else "ns"
    print(f"\n  Kruskal-Wallis: H={kw_stat:.3f}, p={kw_p:.4f} {kw_sig}")

    # Dunn's post-hoc (if significant) 
    if kw_p < 0.05:
        print("  Dunn's post-hoc (Bonferroni corrected):")
        dunn_results = dunn_posthoc(
            sub, group_col="SES_Group", val_col=delta_col, groups=ses_order
        )
        for (g1, g2), p in dunn_results.items():
            sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
            print(f"    {str(g1):<18} vs {str(g2):<18} p={p:.4f} {sig}")
    else:
        print("  No significant difference — post-hoc not run.")

    # Spearman correlation
    spear_r, spear_p = stats.spearmanr(sub["Kupp_Occupation"], sub[delta_col])
    spear_sig = "***" if spear_p < 0.001 else "**" if spear_p < 0.01 else "*" if spear_p < 0.05 else "ns"
    print(f"\n  Spearman (Kupp score vs delta): r={spear_r:.3f}, "
          f"p={spear_p:.4f} {spear_sig}")

    step6_results.append({
        "Variable":     name,
        "n":            n,
        "KW_H":         round(kw_stat, 3),
        "KW_p":         round(kw_p, 4),
        "KW_sig":       kw_sig,
        "Spearman_r":   round(spear_r, 3),
        "Spearman_p":   round(spear_p, 4),
        "Spearman_sig": spear_sig,
    })

#  Summary table 
print(f"\n{'=' * 70}")
print("STEP 6 SUMMARY TABLE")
print(f"{'=' * 70}")
print(f"{'Variable':<22} {'n':>5} {'KW H':>8} {'KW p':>8} {'Sig':>5} "
      f"{'Spearman r':>12} {'Spearman p':>12} {'Sig':>5}")
print("-" * 80)
for r in step6_results:
    print(f"{r['Variable']:<22} {r['n']:>5} {r['KW_H']:>8} {r['KW_p']:>8} "
          f"{r['KW_sig']:>5} {r['Spearman_r']:>12} {r['Spearman_p']:>12} "
          f"{r['Spearman_sig']:>5}")

# Box plots — saved to file only 
colors = ["#d9534f", "#f0ad4e", "#5cb85c"]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, (name, (delta_col, direction)) in enumerate(delta_outcomes.items()):
    sub = df2[["SES_Group", delta_col]].dropna().copy()
    sub["SES_Group"] = sub["SES_Group"].astype(str)

    data_by_group = [
        sub[sub["SES_Group"] == g][delta_col].values
        for g in ses_order
    ]

    ax = axes[i]
    bp = ax.boxplot(data_by_group, tick_labels=ses_order, patch_artist=True)

    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    ax.axhline(0, color="black", linestyle="--", linewidth=0.8, alpha=0.5)
    ax.set_title(f"{name} — Delta by SES Group", fontsize=12, fontweight="bold")
    ax.set_xlabel("SES Group")
    ax.set_ylabel(f"Delta {name} (FU - BL)")
    ax.tick_params(axis="x", rotation=15)

plt.suptitle("Clinical Improvement by SES Group — Active Users",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("step6_ses_vs_deltas.png", dpi=150, bbox_inches="tight")
plt.close()

In [ ]:
# STEP 7: SES, Age, Gender vs Engagement 
from scipy import stats
import matplotlib.pyplot as plt
import numpy as np

print("=" * 70)
print("STEP 7: SES, AGE, GENDER vs ENGAGEMENT")
print("=" * 70)

eng_order     = ["Low", "Moderate", "Champion"]
ses_order     = ["Low (1–4)", "Middle (5–7)", "High (8–10)"]
age_order_str = ["<30", "30–39", "40–49", "50–59", "≥60"]

# Ensure helper columns exist 
if "SES_Group" not in df2.columns:
    df2["SES_Group"] = pd.cut(
        df2["Kupp_Occupation"],
        bins=[0, 4, 7, 10],
        labels=["Low (1–4)", "Middle (5–7)", "High (8–10)"]
    )
df2["SES_Group_str"] = df2["SES_Group"].astype(str)
df2["Age_Group_str"] = df2["Age_Group"].map(age_labels)

step7_results = []

# PART A — Credit_Score as continuous engagement outcome; credit score is a proxy for how often the person engaged with the app.

print("\n" + "═" * 70)
print("PART A — Credit Score (continuous) vs Demographics & SES")
print("═" * 70)

# A1. Spearman: SES & Age vs Credit_Score 
print("\n── A1. Spearman Correlations with Credit Score ──────────────────")
for label, col in [("Kupp_Occupation (SES)", "Kupp_Occupation"),
                    ("Age (years)",           "AGE")]:
    sub  = df2[[col, "Credit_Score"]].dropna()
    r, p = stats.spearmanr(sub[col], sub["Credit_Score"])
    sig  = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    print(f"  {label:<30} r={r:.3f}  p={p:.4f}  {sig}")
    step7_results.append({"Test": f"Spearman: {label} vs Credit_Score",
                           "Stat": round(r,3), "p_value": round(p,4), "Sig": sig})

# A2. Mann-Whitney: Credit_Score by Gender
print("\n── A2. Mann-Whitney: Credit Score by Gender ─────────────────────")
male   = df2[df2["Gender"] == "Male"]["Credit_Score"].dropna()
female = df2[df2["Gender"] == "Female"]["Credit_Score"].dropna()
u, p   = stats.mannwhitneyu(male, female, alternative="two-sided")
sig    = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
print(f"  Male   n={len(male):>4}  mean={male.mean():.2f}  sd={male.std():.2f}")
print(f"  Female n={len(female):>4}  mean={female.mean():.2f}  sd={female.std():.2f}")
print(f"  Mann-Whitney U={u:.1f}  p={p:.4f}  {sig}")
step7_results.append({"Test": "Mann-Whitney: Credit_Score by Gender",
                       "Stat": round(u,1), "p_value": round(p,4), "Sig": sig})

# A3. Kruskal-Wallis: Credit_Score by SES_Group 
print("\n── A3. Kruskal-Wallis: Credit Score by SES Group ────────────────")
ses_grps = [df2[df2["SES_Group_str"] == g]["Credit_Score"].dropna().values
            for g in ses_order]
ses_grps = [g for g in ses_grps if len(g) >= 3]
h, p     = stats.kruskal(*ses_grps)
sig      = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
for g in ses_order:
    grp = df2[df2["SES_Group_str"] == g]["Credit_Score"].dropna()
    print(f"  {g:<18} n={len(grp):>4}  mean={grp.mean():.2f}  sd={grp.std():.2f}")
print(f"  Kruskal-Wallis H={h:.3f}  p={p:.4f}  {sig}")
step7_results.append({"Test": "KW: Credit_Score by SES_Group",
                       "Stat": round(h,3), "p_value": round(p,4), "Sig": sig})

# A4. Kruskal-Wallis: Credit_Score by Age_Group 
print("\n── A4. Kruskal-Wallis: Credit Score by Age Group ────────────────")
age_grps = [df2[df2["Age_Group_str"] == g]["Credit_Score"].dropna().values
            for g in age_order_str]
age_grps = [g for g in age_grps if len(g) >= 3]
h, p     = stats.kruskal(*age_grps)
sig      = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
for g in age_order_str:
    grp = df2[df2["Age_Group_str"] == g]["Credit_Score"].dropna()
    if len(grp) > 0:
        print(f"  {g:<10} n={len(grp):>4}  mean={grp.mean():.2f}  sd={grp.std():.2f}")
print(f"  Kruskal-Wallis H={h:.3f}  p={p:.4f}  {sig}")
step7_results.append({"Test": "KW: Credit_Score by Age_Group",
                       "Stat": round(h,3), "p_value": round(p,4), "Sig": sig})

# Part B — Engagement_category as categorical outcome; here, we treat engagement category as an ordinal variable (Low < Moderate < Champion) and use non-parametric tests to see if it differs

print("\n" + "═" * 70)
print("PART B — Engagement Category vs Demographics & SES")
print("═" * 70)

# B1. Kruskal-Wallis: Engagement_Code by SES_Group
print("\n── B1. Kruskal-Wallis: Engagement Code by SES Group ────────────")
ses_eng = [df2[df2["SES_Group_str"] == g]["Engagement_Code"].dropna().values
           for g in ses_order]
ses_eng = [g for g in ses_eng if len(g) >= 3]
h, p    = stats.kruskal(*ses_eng)
sig     = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
for g in ses_order:
    grp = df2[df2["SES_Group_str"] == g]["Engagement_Code"].dropna()
    print(f"  {g:<18} n={len(grp):>4}  mean={grp.mean():.2f}  sd={grp.std():.2f}")
print(f"  Kruskal-Wallis H={h:.3f}  p={p:.4f}  {sig}")
step7_results.append({"Test": "KW: Engagement_Code by SES_Group",
                       "Stat": round(h,3), "p_value": round(p,4), "Sig": sig})

# B2. Kruskal-Wallis: Engagement_Code by Age_Group
print("\n── B2. Kruskal-Wallis: Engagement Code by Age Group ────────────")
age_eng = [df2[df2["Age_Group_str"] == g]["Engagement_Code"].dropna().values
           for g in age_order_str]
age_eng = [g for g in age_eng if len(g) >= 3]
h, p    = stats.kruskal(*age_eng)
sig     = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
for g in age_order_str:
    grp = df2[df2["Age_Group_str"] == g]["Engagement_Code"].dropna()
    if len(grp) > 0:
        print(f"  {g:<10} n={len(grp):>4}  mean={grp.mean():.2f}  sd={grp.std():.2f}")
print(f"  Kruskal-Wallis H={h:.3f}  p={p:.4f}  {sig}")
step7_results.append({"Test": "KW: Engagement_Code by Age_Group",
                       "Stat": round(h,3), "p_value": round(p,4), "Sig": sig})

# B3. Mann-Whitney: Engagement_Code by Gender 
print("\n── B3. Mann-Whitney: Engagement Code by Gender ──────────────────")
male_eng   = df2[df2["Gender"] == "Male"]["Engagement_Code"].dropna()
female_eng = df2[df2["Gender"] == "Female"]["Engagement_Code"].dropna()
u, p       = stats.mannwhitneyu(male_eng, female_eng, alternative="two-sided")
sig        = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
print(f"  Male   n={len(male_eng):>4}  mean={male_eng.mean():.2f}  sd={male_eng.std():.2f}")
print(f"  Female n={len(female_eng):>4}  mean={female_eng.mean():.2f}  sd={female_eng.std():.2f}")
print(f"  Mann-Whitney U={u:.1f}  p={p:.4f}  {sig}")
step7_results.append({"Test": "Mann-Whitney: Engagement_Code by Gender",
                       "Stat": round(u,1), "p_value": round(p,4), "Sig": sig})

# Part C — Crosstabs with Chi-square, here I treat engagement category as a categorical variable and see if its distribution differs

print("\n" + "═" * 70)
print("PART C — Crosstabs: Engagement Category Distribution")
print("═" * 70)

def print_crosstab_chi2(df, row_col, col_col, row_order, col_order, title):
    print(f"\n── {title} ──")
    sub = df[[row_col, col_col]].dropna().copy()
    sub[row_col] = sub[row_col].astype(str)
    ct  = pd.crosstab(sub[row_col], sub[col_col])
    ct  = ct.reindex(index=[str(r) for r in row_order],
                     columns=col_order, fill_value=0)
    header = f"  {'Group':<20}" + "".join([f"{c:>12}" for c in col_order]) + f"{'Total':>10}"
    print(header)
    print("  " + "-" * (20 + 12 * len(col_order) + 10))
    for grp in row_order:
        grp_str = str(grp)
        if grp_str in ct.index:
            row   = ct.loc[grp_str]
            total = row.sum()
            if total > 0:
                counts = "".join([f"{row[c]:>8}({row[c]/total*100:.0f}%)"
                                  for c in col_order])
            else:
                counts = "".join([f"{'0(0%)':>12}" for c in col_order])
            print(f"  {str(grp):<20}{counts}{total:>10}")
    chi2, p, dof, _ = stats.chi2_contingency(ct)
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    print(f"\n  Chi-square: χ²={chi2:.3f}  df={dof}  p={p:.4f}  {sig}")
    step7_results.append({"Test": f"Chi-square: Engagement × {title}",
                           "Stat": round(chi2,3), "p_value": round(p,4), "Sig": sig})

print_crosstab_chi2(df2, "SES_Group_str", "Engagement_category",
                    ses_order, eng_order,
                    "Engagement × SES Group")

print_crosstab_chi2(df2, "Age_Group_str", "Engagement_category",
                    age_order_str, eng_order,
                    "Engagement × Age Group")

print_crosstab_chi2(df2, "Gender", "Engagement_category",
                    ["Male", "Female"], eng_order,
                    "Engagement × Gender")

# Summary table 
print(f"\n{'=' * 70}")
print("STEP 7 SUMMARY TABLE")
print(f"{'=' * 70}")
print(f"{'Test':<55} {'Stat':>8} {'p-value':>8} {'Sig':>5}")
print("-" * 80)
for r in step7_results:
    print(f"{r['Test']:<55} {r['Stat']:>8} {r['p_value']:>8} {r['Sig']:>5}")

# Plots

colors_ses = ["#d9534f", "#f0ad4e", "#5cb85c"]
colors_eng = ["#d9534f", "#f0ad4e", "#5cb85c"]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

# Plot 1: Credit Score by SES Group 
ax   = axes[0]
data = [df2[df2["SES_Group_str"] == g]["Credit_Score"].dropna().values
        for g in ses_order]
bp   = ax.boxplot(data, tick_labels=ses_order, patch_artist=True)
for patch, color in zip(bp["boxes"], colors_ses):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax.set_title("Credit Score by SES Group", fontweight="bold")
ax.set_xlabel("SES Group"); ax.set_ylabel("Credit Score")
ax.tick_params(axis="x", rotation=15)

# Plot 2: Credit Score by Age Group 
ax   = axes[1]
data = [df2[df2["Age_Group_str"] == g]["Credit_Score"].dropna().values
        for g in age_order_str]
bp   = ax.boxplot(data, tick_labels=age_order_str, patch_artist=True)
for patch in bp["boxes"]:
    patch.set_facecolor("#5bc0de"); patch.set_alpha(0.7)
ax.set_title("Credit Score by Age Group", fontweight="bold")
ax.set_xlabel("Age Group"); ax.set_ylabel("Credit Score")
ax.tick_params(axis="x", rotation=15)

# Plot 3: Credit Score by Gender 
ax   = axes[2]
data = [male.values, female.values]
bp   = ax.boxplot(data, tick_labels=["Male", "Female"], patch_artist=True)
for patch, color in zip(bp["boxes"], ["#5bc0de", "#f0ad4e"]):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax.set_title("Credit Score by Gender", fontweight="bold")
ax.set_xlabel("Gender"); ax.set_ylabel("Credit Score")

# Plot 4: Engagement % by SES Group (stacked bar) 
ax  = axes[3]
sub = df2[["SES_Group_str", "Engagement_category"]].dropna().copy()
ct  = pd.crosstab(sub["SES_Group_str"], sub["Engagement_category"])
ct  = ct.reindex(index=ses_order, columns=eng_order, fill_value=0)
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
bottom = np.zeros(len(ses_order))
for eng, color in zip(eng_order, colors_eng):
    vals = ct_pct[eng].values if eng in ct_pct.columns else np.zeros(len(ses_order))
    ax.bar(ses_order, vals, bottom=bottom, label=eng, color=color, alpha=0.8)
    bottom += vals
ax.set_title("Engagement Category % by SES Group", fontweight="bold")
ax.set_xlabel("SES Group"); ax.set_ylabel("Percentage (%)")
ax.legend(title="Engagement", loc="upper right")
ax.tick_params(axis="x", rotation=15)

# Plot 5: Engagement % by Age Group (stacked bar)
ax  = axes[4]
sub = df2[["Age_Group_str", "Engagement_category"]].dropna().copy()
ct  = pd.crosstab(sub["Age_Group_str"], sub["Engagement_category"])
ct  = ct.reindex(index=age_order_str, columns=eng_order, fill_value=0)
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
bottom = np.zeros(len(age_order_str))
for eng, color in zip(eng_order, colors_eng):
    vals = ct_pct[eng].values if eng in ct_pct.columns else np.zeros(len(age_order_str))
    ax.bar(age_order_str, vals, bottom=bottom, label=eng, color=color, alpha=0.8)
    bottom += vals
ax.set_title("Engagement Category % by Age Group", fontweight="bold")
ax.set_xlabel("Age Group"); ax.set_ylabel("Percentage (%)")
ax.legend(title="Engagement", loc="upper right")
ax.tick_params(axis="x", rotation=15)

# Plot 6: Engagement % by Gender (stacked bar) 
ax  = axes[5]
sub = df2[["Gender", "Engagement_category"]].dropna().copy()
ct  = pd.crosstab(sub["Gender"], sub["Engagement_category"])
ct  = ct.reindex(index=["Male", "Female"], columns=eng_order, fill_value=0)
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
bottom = np.zeros(2)
for eng, color in zip(eng_order, colors_eng):
    vals = ct_pct[eng].values if eng in ct_pct.columns else np.zeros(2)
    ax.bar(["Male", "Female"], vals, bottom=bottom, label=eng, color=color, alpha=0.8)
    bottom += vals
ax.set_title("Engagement Category % by Gender", fontweight="bold")
ax.set_xlabel("Gender"); ax.set_ylabel("Percentage (%)")
ax.legend(title="Engagement", loc="upper right")

plt.suptitle("Engagement vs Demographics & SES — Active Users",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("step7_engagement_vs_demographics.png", dpi=150, bbox_inches="tight")
plt.close()

In [ ]:
# Step 8: Does Engagement mediate SES → Clinical Outcome? 
from scipy import stats
import matplotlib.pyplot as plt
import numpy as np

print("=" * 70)
print("STEP 8: MEDIATION — SES → ENGAGEMENT → CLINICAL OUTCOME")
print("=" * 70)
print("""
  Mediation model (Baron & Kenny):
  Path A : SES → Engagement (Credit_Score)
  Path B : Engagement → Delta Outcome
  Path C : SES → Delta Outcome (total effect)
  Path C': SES → Delta Outcome controlling for Engagement (direct effect)
  Indirect effect = A × B
  If C' < C and significant → mediation supported
""")

delta_outcomes = {
    "HbA1c":             "Delta_HbA1c",
    "HDL":               "Delta_HDL",
    "TGL":               "Delta_TGL",
    "Serum_Cholesterol": "Delta_Serum_Cholesterol",
}

# Bootstrapped indirect effect function 
def bootstrap_indirect(x, m, y, n_boot=1000, ci=95, seed=42):
    """
    Bootstrap indirect effect (a*b) for mediation.
    x = predictor (SES), m = mediator (Engagement), y = outcome (Delta)
    Returns: indirect_effect, lower_ci, upper_ci
    """
    rng = np.random.default_rng(seed)
    n   = len(x)
    indirect_effects = []

    for _ in range(n_boot):
        idx   = rng.integers(0, n, size=n)
        x_b   = x[idx]
        m_b   = m[idx]
        y_b   = y[idx]

        # Path A: x → m
        slope_a, _, _, _, _ = stats.linregress(x_b, m_b)
        # Path B: m → y (controlling for x)
        # Simple: regress y on m and x, get m coefficient
        X_mat  = np.column_stack([np.ones(n), x_b, m_b])
        try:
            coefs  = np.linalg.lstsq(X_mat, y_b, rcond=None)[0]
            slope_b = coefs[2]  # coefficient for m
        except Exception:
            continue

        indirect_effects.append(slope_a * slope_b)

    indirect_effects = np.array(indirect_effects)
    lower = np.percentile(indirect_effects, (100 - ci) / 2)
    upper = np.percentile(indirect_effects, 100 - (100 - ci) / 2)
    point = np.mean(indirect_effects)
    return point, lower, upper

step8_results = []

for name, delta_col in delta_outcomes.items():

    print(f"\n{'─' * 70}")
    print(f"{name}")
    print(f"{'─' * 70}")

    # Prepare data — drop NaN across all three variables 
    sub = df2[["Kupp_Occupation", "Credit_Score", delta_col]].dropna()
    x   = sub["Kupp_Occupation"].values.astype(float)  # SES
    m   = sub["Credit_Score"].values.astype(float)     # Mediator
    y   = sub[delta_col].values.astype(float)          # Outcome
    n   = len(sub)
    print(f"  n = {n}")

    if n < 20:
        print("  Insufficient data, skipping.")
        continue

    # Path A: SES → Engagement (Credit_Score) 
    slope_a, int_a, r_a, p_a, _ = stats.linregress(x, m)
    sig_a = "***" if p_a < 0.001 else "**" if p_a < 0.01 else "*" if p_a < 0.05 else "ns"
    print(f"\n  Path A  (SES → Credit_Score):      β={slope_a:.4f}  p={p_a:.4f}  {sig_a}")

    # Path C: SES → Delta (total effect)
    slope_c, int_c, r_c, p_c, _ = stats.linregress(x, y)
    sig_c = "***" if p_c < 0.001 else "**" if p_c < 0.01 else "*" if p_c < 0.05 else "ns"
    print(f"  Path C  (SES → {name} delta):  β={slope_c:.4f}  p={p_c:.4f}  {sig_c}")

    # Path B & C': multiple regression (SES + Engagement → Delta) 
    X_mat   = np.column_stack([np.ones(n), x, m])
    coefs, _, _, _ = np.linalg.lstsq(X_mat, y, rcond=None)
    int_cp, slope_cp, slope_b = coefs

    # Get p-values via t-test on residuals
    y_pred  = X_mat @ coefs
    resid   = y - y_pred
    mse     = np.sum(resid**2) / (n - 3)
    var_b   = mse * np.linalg.inv(X_mat.T @ X_mat)
    se      = np.sqrt(np.diag(var_b))
    t_vals  = coefs / se
    p_vals  = [2 * stats.t.sf(abs(t), df=n-3) for t in t_vals]

    p_b  = p_vals[2]
    p_cp = p_vals[1]
    sig_b  = "***" if p_b  < 0.001 else "**" if p_b  < 0.01 else "*" if p_b  < 0.05 else "ns"
    sig_cp = "***" if p_cp < 0.001 else "**" if p_cp < 0.01 else "*" if p_cp < 0.05 else "ns"

    print(f"  Path B  (Credit_Score → {name} delta, controlling SES):")
    print(f"          β={slope_b:.4f}  p={p_b:.4f}  {sig_b}")
    print(f"  Path C' (SES → {name} delta, controlling Engagement):")
    print(f"          β={slope_cp:.4f}  p={p_cp:.4f}  {sig_cp}")

    # Indirect effect (bootstrapped)
    indirect, lower, upper = bootstrap_indirect(x, m, y, n_boot=1000)
    sig_indirect = "significant" if not (lower <= 0 <= upper) else "non-significant"
    print(f"\n  Indirect effect (A×B, bootstrapped 95% CI):")
    print(f"    Point estimate = {indirect:.4f}")
    print(f"    95% CI = [{lower:.4f}, {upper:.4f}]  → {sig_indirect}")

    # Mediation conclusion 
    if sig_indirect == "significant" and abs(slope_cp) < abs(slope_c):
        if abs(slope_cp) < 0.001 or p_cp > 0.05:
            conclusion = "Full mediation"
        else:
            conclusion = "Partial mediation"
    else:
        conclusion = "No mediation"

    print(f"\n  ► Conclusion: {conclusion}")

    step8_results.append({
        "Variable":    name,
        "n":           n,
        "Path_A_b":    round(slope_a, 4),
        "Path_A_p":    round(p_a, 4),
        "Path_B_b":    round(slope_b, 4),
        "Path_B_p":    round(p_b, 4),
        "Path_C_b":    round(slope_c, 4),
        "Path_C_p":    round(p_c, 4),
        "Path_Cp_b":   round(slope_cp, 4),
        "Path_Cp_p":   round(p_cp, 4),
        "Indirect":    round(indirect, 4),
        "CI_lower":    round(lower, 4),
        "CI_upper":    round(upper, 4),
        "Conclusion":  conclusion,
    })

# Summary table 
print(f"\n{'=' * 70}")
print("STEP 8 SUMMARY TABLE")
print(f"{'=' * 70}")
print(f"{'Variable':<22} {'n':>5} {'Path A β':>10} {'Path B β':>10} "
      f"{'Path C β':>10} {'Path C β':>10} {'Indirect':>10} {'95% CI':>20} {'Conclusion'}")
print(f"{'':22} {'':5} {'(SES→Eng)':>10} {'(Eng→Out)':>10} "
      f"{'(total)':>10} {'(direct)':>10} {'(A×B)':>10} {'':>20}")
print("-" * 115)
for r in step8_results:
    ci  = f"[{r['CI_lower']}, {r['CI_upper']}]"
    print(f"{r['Variable']:<22} {r['n']:>5} {r['Path_A_b']:>10} {r['Path_B_b']:>10} "
          f"{r['Path_C_b']:>10} {r['Path_Cp_b']:>10} {r['Indirect']:>10} "
          f"{ci:>20} {r['Conclusion']}")

# Path diagram plot 
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, r in enumerate(step8_results):
    ax = axes[i]
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 6)
    ax.axis("off")

    # Nodes
    node_style = dict(boxstyle="round,pad=0.5", facecolor="#dce9f5",
                      edgecolor="#2c6fad", linewidth=2)
    ax.text(1, 3, "SES\n(Kupp)", ha="center", va="center",
            fontsize=10, fontweight="bold", bbox=node_style)
    ax.text(5, 5.2, "Engagement\n(Credit Score)", ha="center", va="center",
            fontsize=10, fontweight="bold", bbox=node_style)
    ax.text(9, 3, f"{r['Variable']}\nDelta", ha="center", va="center",
            fontsize=10, fontweight="bold", bbox=node_style)

    arrow_style = dict(arrowstyle="->", color="#2c6fad", lw=2)

    # Path A: SES → Engagement
    ax.annotate("", xy=(4.1, 4.9), xytext=(1.7, 3.4),
                arrowprops=arrow_style)
    sig_a = "***" if r["Path_A_p"] < 0.001 else "**" if r["Path_A_p"] < 0.01 else "*" if r["Path_A_p"] < 0.05 else "ns"
    ax.text(2.7, 4.4, f"A: β={r['Path_A_b']}\np={r['Path_A_p']} {sig_a}",
            ha="center", fontsize=8, color="#2c6fad")

    # Path B: Engagement → Outcome
    ax.annotate("", xy=(8.2, 3.4), xytext=(5.9, 4.9),
                arrowprops=arrow_style)
    sig_b = "***" if r["Path_B_p"] < 0.001 else "**" if r["Path_B_p"] < 0.01 else "*" if r["Path_B_p"] < 0.05 else "ns"
    ax.text(7.3, 4.4, f"B: β={r['Path_B_b']}\np={r['Path_B_p']} {sig_b}",
            ha="center", fontsize=8, color="#2c6fad")

    # Path C' (direct): SES → Outcome
    ax.annotate("", xy=(8.2, 2.8), xytext=(1.8, 2.8),
                arrowprops=arrow_style)
    sig_cp = "***" if r["Path_Cp_p"] < 0.001 else "**" if r["Path_Cp_p"] < 0.01 else "*" if r["Path_Cp_p"] < 0.05 else "ns"
    ax.text(5, 2.3, f"C' (direct): β={r['Path_Cp_b']}  p={r['Path_Cp_p']} {sig_cp}",
            ha="center", fontsize=8, color="#555")

    # Indirect effect
    ci_str = f"[{r['CI_lower']}, {r['CI_upper']}]"
    ax.text(5, 1.4, f"Indirect (A×B): {r['Indirect']}  95% CI {ci_str}",
            ha="center", fontsize=8, color="#c0392b")

    # Conclusion
    color = "#27ae60" if "mediation" in r["Conclusion"].lower() else "#c0392b"
    ax.text(5, 0.7, f"► {r['Conclusion']}", ha="center",
            fontsize=10, fontweight="bold", color=color)

    ax.set_title(f"Mediation: SES → Engagement → {r['Variable']}",
                 fontsize=11, fontweight="bold")

plt.suptitle("Mediation Analysis — Active Users", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("step8_mediation.png", dpi=150, bbox_inches="tight")
plt.close() 